In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from scipy.optimize import line_search

spline_points = np.array([[0,1], [0.33, 0.33], [0.6, .5], [0.66, 0], [0.9, 1.75], [1,2]])

objective = CubicSpline(*spline_points.T, bc_type="natural")
objective_derivative = objective.derivative()

MT_line_search_samples = list()

def logged_objective(x):
    print(x)
    if x <= 1:
        MT_line_search_samples.append(x)
    return objective(x)

# Generate data for plotting
x_vals = np.linspace(0, 1, 200)
y_vals = [objective(x) for x in x_vals]

derivative_vals = [objective_derivative(x) for x in x_vals]

line_search(logged_objective, objective_derivative, 0, 1, amax=1, c1=0.1, c2=0.2)

# Plot the function
plt.figure(figsize=(10, 6))
plt.plot(x_vals, y_vals, label="Piecewise Polynomial", color="blue")
plt.scatter(MT_line_search_samples, [objective(x) for x in MT_line_search_samples])
plt.axhline(0, color="black", linewidth=0.5, linestyle="--")
plt.axvline(0, color="black", linewidth=0.5, linestyle="--")
plt.title("Piecewise Polynomial Function")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
from bayesian_line_search.line_search import gp_line_search, LineSearchDebugOptions

spline_points = np.array([[0,1], [0.33, 0.33], [0.6, .5], [0.66, 0], [0.9, 1.75], [1,2]])

objective = CubicSpline(*spline_points.T, bc_type="natural")
objective_derivative = objective.derivative()

line_search_samples = list()

def logged_objective(x):
    print(x)
    if x <= 1:
        line_search_samples.append(x)
    return objective(x), objective_derivative(x)

def wolfe_met(x):
    return False

# Generate data for plotting
x_vals = np.linspace(0, 1, 500)
y_vals = [objective(x) for x in x_vals]

derivative_vals = [objective_derivative(x) for x in x_vals]

gp_line_search(logged_objective, [0, 1], [0,1], wolfe_met, np, LineSearchDebugOptions(), 10)

# Plot the function
plt.figure(figsize=(10, 6))
plt.plot(x_vals, y_vals, label="Piecewise Polynomial", color="blue")
plt.scatter(line_search_samples, [objective(x) for x in line_search_samples])
plt.axhline(0, color="black", linewidth=0.5, linestyle="--")
plt.axvline(0, color="black", linewidth=0.5, linestyle="--")
plt.title("Piecewise Polynomial Function")
plt.xlabel("x")
plt.ylabel("f(x)")
plt.grid(alpha=0.3)
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import gaussian_process.GPfunctions as gp
from gaussian_process import GaussianProcess
from gaussian_process.kernels import Matern2_5Kernel, SquaredExponentialKernel
from acquisition.acquisition import LowerConfidenceBound

next_sample = line_search_samples[6]
line_search_samples = line_search_samples[:6]
line_search_sample_fs = [objective(x) for x in line_search_samples]
line_search_sample_gs = [objective_derivative(x) for x in line_search_samples]

kernel = Matern2_5Kernel(1. / (len(line_search_samples) - 1),)

GP_posterior = GaussianProcess(
    kernel=kernel,
    x_known=line_search_samples,
    f_known=line_search_sample_fs,
    g_known=line_search_sample_gs,
    f_noise=1e-10,
    g_noise=1e-10,
)

acquisition_function = LowerConfidenceBound(GP_posterior, 2)

# Calculate derivative of mean
mean, variance = GP_posterior(x_vals)
std = GP_posterior.std_deviation(x_vals, variance=variance)

lcb_values = acquisition_function(x_vals)

fig, (ax1) = plt.subplots(1, 1)
ax1.plot(x_vals, y_vals, label="Objective", linestyle="dotted")
ax1.scatter(line_search_samples, line_search_sample_fs, label="Observations")
ax1.scatter(MT_line_search_samples, [objective(x) for x in MT_line_search_samples], label="MT Observations", color='red')
ax1.plot(x_vals, mean, label="mean+-std", color='orange')
ax1.plot(x_vals, mean - std * 2, linestyle='--', color='orange')
ax1.plot(x_vals, mean + std * 2, linestyle='--', color='orange')
ax1.plot(x_vals, lcb_values, label="LCB", color='green')
ax1.legend()
ax1.grid()
ax1.set(xlabel="$x$", ylabel="$f(x)$")
ax1.label_outer()
plt.show()

In [ ]:
print(f"Next sample: ({next_sample},{objective(next_sample)})")
print(f"Acquisition: ({next_sample},{acquisition_function(next_sample).item()})")

In [ ]:
print("line_search_samples (x, y) pairs: " + " ".join([f"({x},{objective(x)})" for x in line_search_samples]))
print("MT_line_search_samples (x, y) pairs: " + " ".join([f"({x},{objective(x)})" for x in MT_line_search_samples]))

In [13]:
import csv

with open("plot.csv", "w") as f:
    writer = csv.writer(f)
    writer.writerow(("x", "y", "mean", "std_upper", "std_lower", "lcb"))
    for row in zip(x_vals, y_vals, mean, mean + std * 2, mean - std * 2, lcb_values):
        writer.writerow(row)